<a href="https://colab.research.google.com/github/harshaRathnayaka/FlaskWithAndroid/blob/master/notebooks/04_agent_tool_call_web_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agent Tool-Call Web Research

Give an agent an annotated search function so it can retrieve web results before writing its summary.

## 1. Install dependencies

In [1]:
%pip install -q openai-agents ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 35.6 MB/s eta 0:00:00


## 2. Choose a model provider

Set `PROVIDER` to `"openai"` or `"gemini"`, then add the corresponding API key to Colab Secrets.

In [2]:
import os

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

PROVIDER = "gemini"  # Change to "openai" to use OpenAI.
OPENAI_MODEL_NAME = "gpt-5-mini"
GEMINI_MODEL_NAME = "gemini-3.8-flash"

if PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    if not os.environ["OPENAI_API_KEY"]:
        raise ValueError("Add OPENAI_API_KEY to Colab Secrets.")
    model = OPENAI_MODEL_NAME
elif PROVIDER == "gemini":
    GEMINI_API_KEY = userdata.get("GeminiAPI")
    if not GEMINI_API_KEY:
        raise ValueError("Add GEMINI_API_KEY to Colab Secrets.")
    set_tracing_disabled(disabled=True)
    model = OpenAIChatCompletionsModel(
        model=GEMINI_MODEL_NAME,
        openai_client=AsyncOpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=GEMINI_API_KEY,
        ),
    )
else:
    raise ValueError("PROVIDER must be 'openai' or 'gemini'.")

## 3. Define the agent tool

The `@function_tool` annotation makes `search_web` available for the agent to call.

In [3]:
from agents import function_tool
from ddgs import DDGS
from ddgs.exceptions import DDGSException


@function_tool
def search_web(query: str) -> str:
    """Search the web and return source titles, summaries, and URLs."""
    try:
        results = DDGS(timeout=15).text(query, max_results=5)
    except DDGSException as error:
        print(f"Search failed: {error}")
        return "Search is temporarily unavailable."

    print(f"Search query: {query}")
    print(f"Results found: {len(results)}")
    for index, result in enumerate(results, start=1):
        print(f"\n{index}. {result.get('title', 'Untitled source')}")
        print(result.get('body', 'No summary available.'))
        print(result.get('href', 'No URL available.'))

    return "\n\n".join(
        f"{result.get('title', 'Untitled source')}\n{result.get('body', '')}\n{result.get('href', '')}"
        for result in results
    )


@function_tool
def print_completion_message() -> str:
    """Print a message when research is complete and the final answer is ready."""
    message = "Research complete. Preparing the final answer."
    print(message)
    return message

## 4. Let the agent search and summarize

In [5]:
from agents import Agent, Runner

agent = Agent(
    name="Research Assistant",
    instructions=(
        "Always call search_web before answering. After you have gathered the research, "
        "call print_completion_message immediately before your final answer. Summarize only "
        "the search results and end with a Sources section containing the URLs you used."
    ),
    tools=[search_web, print_completion_message],
    model=model,
)

question = "What is agentic AI, and how is it used in business?"
result = await Runner.run(starting_agent=agent, input=question)

print(result.final_output)

Search query: what is agentic AI business use cases definition
Results found: 5

1. What Is Agentic AI? Definition, Benefits, Use Cases - domo.com
Jun 1, 2026 · Learn what agentic AI is and how autonomous agents plan, use tools, and stay governed. Get benefits, use cases, risks, and agentic AI vs generative AI.
https://www.domo.com/blog/agentic-ai-explained-definition-benefits-and-use-cases

2. Agentic AI, explained - MIT Sloan
Feb 18, 2026 · What agentic AI is and how it differs from traditional generative AI tools like chatbots. How organizations are already using AI agents to automate complex, multistep workflows.
https://mitsloan.mit.edu/ideas-made-to-matter/agentic-ai-explained

3. What Is Agentic AI? Definition, 6 Levels & Examples (2026)
1 day ago · Agentic AI acts on its own — planning, using tools, and adapting until a task is done. Learn how it differs from chatbots, the 6 levels of autonomy, and real tools to try.
https://agentic.ai/what-is-agentic-ai

4. What is Agentic AI?

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 12.950184214s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.8-flash'}, '